# Semaine 8 — Jour 4 : Coûts, budgets et arbitrages

Calculer le coût des tokens entrants, cachés et sortants sans double comptage.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour le parcours complet avec entretien et challenge. J1 conserve sa durée et son contenu d’origine; il peut être réparti sur plusieurs sessions.

Prérequis : J1, Python 3.11+, JSON et tests backend. Aucun accès fournisseur, aucune clé LLM et aucune infrastructure externe nécessaires.

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks : `notebooks/week08/S8_J4_couts.ipynb` et `notebooks/week08/teacher/S8_J4_couts_teacher.ipynb` depuis la racine du dépôt.

Critère de réussite : Simuler une admission par réservation avant l’appel.

Les observations et prix sont synthétiques; les résultats mesurent le fonctionnement du protocole local, pas un fournisseur réel.


# Objectifs mesurables

- Calculer le coût des tokens entrants, cachés et sortants sans double comptage.
- Inclure les échecs et reprises dans le coût par résultat utile.
- Simuler une admission par réservation avant l’appel.

## Preuves attendues

Une sortie reproductible, une interprétation de son dénominateur et une limite opérationnelle correctement expliquée.


# S8 J4 — Coûts, budgets et arbitrages


## 1. Donner un sens au coût unitaire
Un coût par requête ne dit pas combien coûte une résolution utile. Si une réponse échoue et nécessite deux reprises, les trois appels peuvent être facturés pour un seul résultat. Il faut préciser l’unité : tentative, demande, conversation ou tâche métier achevée. Le lab additionne trente-six tentatives et divise par le nombre d’essais techniquement réussis associés à un cas fonctionnel réussi.

Cette définition est locale : elle réutilise l’étiquette de qualité de J1 pour les trois essais de chaque cas. Elle ne prouve pas que trois générations indépendantes seraient correctes. La baseline totalise trente succès utiles selon ce proxy; le candidat trente-deux. Un timeout reste facturé dans la fixture, avec les tokens synthétiques fournis.

## 2. Calculer sans double comptage
Le compteur input_tokens inclut cached_input_tokens. Il faut soustraire le cache avant d’appliquer le tarif normal. La formule est : coût = ((entrée−cache) × prix_entrée + cache × prix_cache + sortie × prix_sortie) / 1000000. Les prix du cours sont inventés, nommés synthetic et exprimés en USD par million de tokens. Ils ne décrivent aucun tarif commercial.

Le module utilise Decimal avec des valeurs décimales sous forme de chaînes. Cela évite de construire un décimal à partir d’un flottant binaire déjà arrondi. Additionner avant d’arrondir les résultats d’affichage. Valider les compteurs et refuser le cache supérieur au total entrant. Un coût négatif ne doit jamais améliorer une décision.

## 3. Lire le rapport
La baseline coûte 0,06084 USD pour 36 tentatives, soit 0,002028 USD par succès utile selon le proxy. Le candidat coûte 0,05004 USD et 0,00156375 USD par succès utile. Cette amélioration n’autorise pas la livraison : c09 reste un échec critique.

Le coût complet d’un système inclut aussi outils, stockage, observabilité, réseau, calcul et intervention humaine. Le lab calcule seulement des tokens avec une grille synthétique. Si une tâche ne produit aucun succès, usd_per_success vaut null; ce n’est ni zéro ni un succès économique. La décision intégrée refuse ce cas.

## 4. Observer et limiter sont deux fonctions différentes
Un rapport de coût après l’appel ne bloque pas une dépense. Le composant Budget réserve un montant maximal avant l’admission, puis règle le coût observé. Un budget de 0,01 autorise une réservation de 0,008 mais refuse une deuxième réservation simultanée du même montant. Après règlement à 0,003, une partie du plafond redevient disponible.

La réservation suppose que la borne maximale soit crédible : limite de sortie, nombre d’appels et politique de retries doivent être maîtrisés. Le code refuse un règlement supérieur au plafond pour signaler une incohérence; il ne peut pas annuler une facture fournisseur déjà engagée. Le ledger est en mémoire et séquentiel, sans garantie entre processus ou après crash.

## 5. Arbitrer sans supprimer les obligations
Réduire le contexte, mettre en cache, changer de modèle ou borner les reprises peut réduire les coûts. Chaque modification demande une nouvelle évaluation et une mesure comparable. Éviter de tronquer le contexte métier nécessaire ou de retirer les refus de sécurité pour obtenir un meilleur score financier.

Des règles simples peuvent router les tâches faciles vers une option moins chère. Il faut cependant mesurer les erreurs de routage et le coût des escalades. Une politique de fallback multiplie parfois les appels. L’objectif n’est pas le prix du premier appel, mais le coût d’un résultat conforme au contrat.

## À retenir
Le coût utile garde les échecs dans le numérateur et définit le succès dans le dénominateur. Le budget d’admission complète la mesure comptable; il ne remplace pas la qualité ni la sécurité.


# Lab exécutable

Cellules autonomes à partir du dépôt; aucune requête réseau.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week08/ops.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from decimal import Decimal
from examples.week08.demo import load
from examples.week08.ops import cost_report, Budget
data = load("baseline")
passed = {c["id"] for c in data["evaluation"]["cases"] if c["passed"]}
report = cost_report(data["runs"], data["prices"], passed)
assert Decimal(report["total_usd"]) == Decimal("0.06084")
assert report["successful_attempts"] == 30
assert Decimal(report["usd_per_success"]) == Decimal("0.002028")
print(report)

# Exercices

Conserver une première réponse avant de lire la correction.

## Exercice 1 — Calculer un appel

Avec 600 tokens entrants dont 100 cachés, 80 sortants et les prix 2 / 0,5 / 8 USD par million, calculer le coût.

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.

## Exercice 2 — Évaluer une reprise

Pourquoi une tentative échouée doit-elle rester dans le coût total ?

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.

## Exercice 3 — Tester le budget

Sur une limite de 0,01, réserver 0,008 deux fois, régler le premier appel à 0,003 puis réessayer une réservation de 0,007.

Livrable : une réponse justifiée, avec calcul ou observation lorsque pertinent.


# Entretien

Répondre en deux minutes par question, avec un exemple et une limite.

## Question 1

Pourquoi préférer Decimal ici ?

## Question 2

Un quota mensuel visible dans un dashboard suffit-il ?


# Challenge

Ajouter un test de règlement répété : le même coût ne doit pas doubler spent. Un coût contradictoire pour la même réservation doit être refusé.

## Critères

Reproductibilité, garde vérifiée, distinction entre résultat mesuré et garantie non démontrée.


# Corrigé des exercices

## Exercice 1

(500×2 +100×0,5 +80×8)/1000000 =0,00169 USD. Ne pas compter les 100 tokens cachés une deuxième fois au tarif normal.

## Exercice 2

Le fournisseur peut facturer du travail même si le résultat est inutilisable ou si la réponse se perd. Exclure l’échec du coût donnerait un coût utile artificiellement bas.

## Exercice 3

La première réservation réussit, la deuxième échoue. Après règlement, spent=0,003 et aucune réservation ne reste. La nouvelle réservation de 0,007 réussit exactement à la limite.


# Réponses d’entretien

## Question 1

Pour représenter exactement les montants décimaux de la grille et reporter l’arrondi à l’affichage. Cela ne corrige pas des compteurs faux ou une formule de facturation incomplète.

## Question 2

Non : c’est une observation après consommation. Une admission doit réserver avant l’appel, borner les sorties/reprises et tenir compte des appels en vol.


# Solution du challenge

Budget conserve les règlements terminés par identifiant. settle(call,0.003) puis le même appel laisse spent=0.003; settle(call,0.004) déclenche EvidenceError. Pour un système distribué, utiliser un ledger durable et une transaction/clé d’idempotence, pas un simple dictionnaire partagé.

## Vérification exécutable

Les cellules supposent que le lab élève a déjà été exécuté.

In [ ]:
from examples.week08.ops import EvidenceError
budget = Budget("0.01")
assert budget.reserve("call-1", "0.008")
assert not budget.reserve("call-2", "0.008")
budget.settle("call-1", "0.003")
budget.settle("call-1", "0.003")
assert budget.spent == Decimal("0.003")
assert budget.reserve("call-3", "0.007")
try:
    budget.settle("call-1", "0.004")
except EvidenceError:
    print("Règlement contradictoire bloqué")
else:
    raise AssertionError("Double règlement accepté")

# Revue pédagogique — J4

## Acquis

- Calculer le coût des tokens entrants, cachés et sortants sans double comptage.
- Inclure les échecs et reprises dans le coût par résultat utile.
- Simuler une admission par réservation avant l’appel.

## Barème indicatif sur 10

- 3 : concepts et vocabulaire correctement reliés au problème.
- 3 : reproduction du lab et lecture des résultats.
- 2 : exercices argumentés.
- 2 : challenge et limite opérationnelle.

Le parcours essentiel peut être terminé sans le challenge. Une confusion entre fixture synthétique et mesure de modèle réel doit être corrigée avant la validation du projet. Vérifier en particulier : La première réservation réussit, la deuxième échoue. Après règlement, spent=0,003 et aucune réservation ne reste. La nouvelle réservation de 0,007 réussit exactement à la limite.


# Références — J4

Sources de référence consultées le 20 octobre 2026. Le lab implémente ses propres contrats pédagogiques; il n’affirme pas déployer les outils cités.

- [OpenTelemetry : concepts d’observabilité](https://opentelemetry.io/docs/concepts/observability-primer/) — signaux et diagnostic.
- [OpenTelemetry : traces](https://opentelemetry.io/docs/concepts/signals/traces/) — contexte et spans pour une intégration réelle.
- [OWASP : injection de prompt, édition 2025](https://genai.owasp.org/llmrisk/llm01-prompt-injection/) — référence datée, sans prétendre être la dernière édition.
- [OWASP : excessive agency, édition 2025](https://genai.owasp.org/llmrisk/llm062025-excessive-agency/) — permissions et autonomie.
- [Python : Decimal](https://docs.python.org/3/library/decimal.html) — calcul décimal pour les coûts.
- [Python : unittest](https://docs.python.org/3/library/unittest.html) — reproduction des contrôles.

Lire prioritairement les sources liées au thème du jour. Les seuils, tarifs et identités du lab sont fictifs et ne proviennent pas de ces documents. Le calcul nearest-rank est défini explicitement dans le chapitre et le code.
